In [0]:

# ============================================================
# CELL 1 — IMPORTS, PARAMETERS, AND CONFIGURATION
# Products Bronze Ingestion
# ============================================================

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from datetime import datetime, timezone
import hashlib

# ------------------------------------------------------------
# Storage paths — preserve the existing project layout
# ------------------------------------------------------------

STORAGE_ACCOUNT = "nestlesalesdatalake"
CONTAINER = "customer-sales-datalake"

BASE_PATH = (
    f"abfss://{CONTAINER}@"
    f"{STORAGE_ACCOUNT}.dfs.core.windows.net/"
)

LANDING_PATH = BASE_PATH + "landing/products/"
BRONZE_PATH = BASE_PATH + "bronze/products/"
MANIFEST_PATH = BASE_PATH + "metadata/file_manifest/"

# ------------------------------------------------------------
# Dataset configuration
# ------------------------------------------------------------

DATASET_NAME = "products"
SOURCE_SYSTEM = "product_api"

# We will confirm these source columns in the next cell.
# Do not ingest data until the actual schema is verified.
REQUIRED_SOURCE_COLUMNS = None
BRONZE_COLUMNS = None

print("Dataset:", DATASET_NAME)
print("Source system:", SOURCE_SYSTEM)
print("Landing path:", LANDING_PATH)
print("Bronze path:", BRONZE_PATH)
print("Manifest path:", MANIFEST_PATH)

print("\nConfiguration initialized successfully.")


Dataset: products
Source system: product_api
Landing path: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/
Bronze path: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/products/
Manifest path: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/metadata/file_manifest/

Configuration initialized successfully.


In [0]:

# ============================================================
# CELL 2 — RUN MODE CONFIGURATION
# ============================================================

# Remove existing widgets first so reruns are predictable
for widget_name in [
    "run_mode",
    "backfill_start_date",
    "backfill_end_date",
    "reprocess_successful"
]:
    try:
        dbutils.widgets.remove(widget_name)
    except Exception:
        pass

# Run mode
dbutils.widgets.dropdown(
    "run_mode",
    "INCREMENTAL",
    ["INCREMENTAL", "BACKFILL", "FULL_LOAD"],
    "Run Mode"
)

# Backfill date range
dbutils.widgets.text(
    "backfill_start_date",
    "",
    "Backfill Start Date (YYYY-MM-DD)"
)

dbutils.widgets.text(
    "backfill_end_date",
    "",
    "Backfill End Date (YYYY-MM-DD)"
)

# Explicit opt-in for replaying successful files
dbutils.widgets.dropdown(
    "reprocess_successful",
    "false",
    ["false", "true"],
    "Reprocess Successful Files?"
)

# Read configuration
RUN_MODE = dbutils.widgets.get("run_mode").strip().upper()

BACKFILL_START_DATE = (
    dbutils.widgets.get("backfill_start_date").strip()
)

BACKFILL_END_DATE = (
    dbutils.widgets.get("backfill_end_date").strip()
)

REPROCESS_SUCCESSFUL = (
    dbutils.widgets.get("reprocess_successful").strip().lower()
    == "true"
)

# Validate run mode
if RUN_MODE not in {"INCREMENTAL", "BACKFILL", "FULL_LOAD"}:
    raise ValueError(f"Unsupported run mode: {RUN_MODE}")

# Validate backfill dates only when BACKFILL is selected
if RUN_MODE == "BACKFILL":
    if not BACKFILL_START_DATE or not BACKFILL_END_DATE:
        raise ValueError(
            "BACKFILL requires both start and end dates."
        )

    from datetime import date

    try:
        start_date = date.fromisoformat(BACKFILL_START_DATE)
        end_date = date.fromisoformat(BACKFILL_END_DATE)
    except ValueError:
        raise ValueError(
            "Backfill dates must use YYYY-MM-DD format."
        )

    if start_date.isoformat() != BACKFILL_START_DATE:
        raise ValueError("Invalid backfill start date format.")

    if end_date.isoformat() != BACKFILL_END_DATE:
        raise ValueError("Invalid backfill end date format.")

    if start_date > end_date:
        raise ValueError(
            "Backfill start date cannot be after end date."
        )

print("========== RUN CONFIGURATION ==========")
print("Dataset:", DATASET_NAME)
print("Run mode:", RUN_MODE)
print("Backfill start:", BACKFILL_START_DATE or "N/A")
print("Backfill end:", BACKFILL_END_DATE or "N/A")
print("Reprocess successful files:", REPROCESS_SUCCESSFUL)


========== RUN CONFIGURATION ==========
Dataset: products
Run mode: INCREMENTAL
Backfill start: N/A
Backfill end: N/A
Reprocess successful files: False


In [0]:

# ============================================================
# CELL 3 — VALIDATE RUN CONFIGURATION
# ============================================================

from datetime import date

if RUN_MODE == "BACKFILL":
    start_date = date.fromisoformat(BACKFILL_START_DATE)
    end_date = date.fromisoformat(BACKFILL_END_DATE)

    if start_date > end_date:
        raise ValueError(
            "Backfill start date cannot be after end date."
        )

    print(
        f"Backfill range: {start_date.isoformat()} "
        f"to {end_date.isoformat()}"
    )

elif RUN_MODE == "FULL_LOAD":
    print("Full Load selected: all eligible files in scope.")

else:
    print("Incremental selected: newly eligible files only.")

print("Run configuration validation passed.")


Incremental selected: newly eligible files only.
Run configuration validation passed.


In [0]:

# ============================================================
# CELL 4 — RUN-MODE-AWARE FILE SELECTION
# Read-only; no claims or writes
# ============================================================

from pyspark.sql import functions as F

manifest_df = spark.read.format("delta").load(MANIFEST_PATH)

eligible_df = manifest_df.filter(
    F.col("dataset") == DATASET_NAME
)

# Never select files already actively claimed by another run
eligible_df = eligible_df.filter(
    F.col("status") != "PROCESSING"
)

if RUN_MODE == "INCREMENTAL":
    # Normal daily processing: new and retryable files
    eligible_df = eligible_df.filter(
        F.col("status").isin("NEW", "FAILED")
    )

elif RUN_MODE == "BACKFILL":
    # Historical files in the requested inclusive date range
    eligible_df = eligible_df.filter(
        F.col("data_date").cast("date").between(
            F.lit(BACKFILL_START_DATE).cast("date"),
            F.lit(BACKFILL_END_DATE).cast("date")
        )
    )

    if not REPROCESS_SUCCESSFUL:
        eligible_df = eligible_df.filter(
            F.col("status").isin("NEW", "FAILED")
        )
    else:
        eligible_df = eligible_df.filter(
            F.col("status").isin("NEW", "FAILED", "SUCCESS")
        )

elif RUN_MODE == "FULL_LOAD":
    # Full manifest scope, with successful-file replay opt-in
    if not REPROCESS_SUCCESSFUL:
        eligible_df = eligible_df.filter(
            F.col("status").isin("NEW", "FAILED")
        )
    else:
        eligible_df = eligible_df.filter(
            F.col("status").isin("NEW", "FAILED", "SUCCESS")
        )

else:
    raise ValueError(f"Unsupported run mode: {RUN_MODE}")

selected_files_df = (
    eligible_df
    .select(
        "dataset",
        "file_name",
        "file_path",
        "data_date",
        "status",
        "content_hash",
        "retry_count"
    )
    .dropDuplicates(["dataset", "file_name"])
    .orderBy("data_date", "file_name")
)

print("Run mode:", RUN_MODE)
print("Reprocess successful:", REPROCESS_SUCCESSFUL)
print("Eligible Products files:", selected_files_df.count())

display(selected_files_df)


Run mode: INCREMENTAL
Reprocess successful: False
Eligible Products files: 0


dataset,file_name,file_path,data_date,status,content_hash,retry_count


In [0]:

# ============================================================
# CELL 5 — VALIDATE SELECTED FILES
# Read-only; no manifest updates
# ============================================================

from pyspark.sql import functions as F

required_columns = [
    "dataset",
    "file_name",
    "file_path",
    "data_date",
    "status",
    "content_hash",
    "retry_count"
]

missing_columns = [
    column
    for column in required_columns
    if column not in selected_files_df.columns
]

if missing_columns:
    raise ValueError(
        f"Selected files DataFrame is missing columns: {missing_columns}"
    )

duplicate_files_df = (
    selected_files_df
    .groupBy("dataset", "file_name")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_files_df.limit(1).count() > 0:
    display(duplicate_files_df)
    raise ValueError("Duplicate file selections detected.")

invalid_files_df = selected_files_df.filter(
    F.col("file_name").isNull()
    | F.col("file_path").isNull()
    | F.col("content_hash").isNull()
    | F.col("status").isNull()
)

if invalid_files_df.limit(1).count() > 0:
    display(invalid_files_df)
    raise ValueError(
        "Selected files have missing required metadata."
    )

print("Run mode:", RUN_MODE)
print("Selected files:", selected_files_df.count())
print("Duplicate selections: 0")
print("Selected-file validation passed.")


Run mode: INCREMENTAL
Selected files: 0
Duplicate selections: 0
Selected-file validation passed.


In [0]:

# ============================================================
# CELL 6 — PREPARE CLAIM CANDIDATES
# Read-only; no manifest updates
# ============================================================

import uuid
from pyspark.sql import functions as F

# Generate a new ID for this execution
EXECUTION_RUN_ID = str(uuid.uuid4())

claim_candidates_df = (
    selected_files_df
    .withColumn("execution_run_id", F.lit(EXECUTION_RUN_ID))
    .withColumn("candidate_status", F.lit("READY_TO_CLAIM"))
)

print("Run mode:", RUN_MODE)
print("Execution run ID:", EXECUTION_RUN_ID)
print("Files ready to claim:", claim_candidates_df.count())

display(claim_candidates_df)


Run mode: INCREMENTAL
Execution run ID: cf95d1a4-f352-4819-8a70-732674d8858d
Files ready to claim: 0


dataset,file_name,file_path,data_date,status,content_hash,retry_count,execution_run_id,candidate_status


In [0]:

# ============================================================
# CELL 7 — PREVIEW ALL RUN MODES
# Read-only; does not claim or process files
# ============================================================

from pyspark.sql import functions as F

manifest_df = spark.read.format("delta").load(MANIFEST_PATH)

products_manifest_df = manifest_df.filter(
    F.col("dataset") == DATASET_NAME
)

def preview_run_mode(mode, reprocess_successful=False):
    candidate_df = products_manifest_df.filter(
        F.col("status") != "PROCESSING"
    )

    if mode == "INCREMENTAL":
        candidate_df = candidate_df.filter(
            F.col("status").isin("NEW", "FAILED")
        )

    elif mode == "BACKFILL":
        if not BACKFILL_START_DATE or not BACKFILL_END_DATE:
            print(
                "BACKFILL preview skipped: enter a start and end date."
            )
            return

        candidate_df = candidate_df.filter(
            F.col("data_date").cast("date").between(
                F.lit(BACKFILL_START_DATE).cast("date"),
                F.lit(BACKFILL_END_DATE).cast("date")
            )
        )

        allowed_statuses = (
            ["NEW", "FAILED", "SUCCESS"]
            if reprocess_successful
            else ["NEW", "FAILED"]
        )

        candidate_df = candidate_df.filter(
            F.col("status").isin(allowed_statuses)
        )

    elif mode == "FULL_LOAD":
        allowed_statuses = (
            ["NEW", "FAILED", "SUCCESS"]
            if reprocess_successful
            else ["NEW", "FAILED"]
        )

        candidate_df = candidate_df.filter(
            F.col("status").isin(allowed_statuses)
        )

    print(
        f"{mode}: {candidate_df.count()} eligible manifest records"
    )

preview_run_mode("INCREMENTAL")
preview_run_mode("BACKFILL", reprocess_successful=False)
preview_run_mode("FULL_LOAD", reprocess_successful=False)


INCREMENTAL: 0 eligible manifest records
BACKFILL preview skipped: enter a start and end date.
FULL_LOAD: 0 eligible manifest records


In [0]:

# ============================================================
# CELL 8 — PRODUCTS MANIFEST INVENTORY
# Read-only; Serverless-compatible
# ============================================================

from pyspark.sql import functions as F

manifest_df = spark.read.format("delta").load(MANIFEST_PATH)

products_inventory_df = (
    manifest_df
    .filter(F.col("dataset") == DATASET_NAME)
    .groupBy("data_date", "status")
    .agg(
        F.count("*").alias("file_count"),
        F.countDistinct("file_name").alias("distinct_files")
    )
    .orderBy("data_date", "status")
)

display(products_inventory_df)

print("Total Products manifest records:",
      manifest_df.filter(
          F.col("dataset") == DATASET_NAME
      ).count())

print("Products landing path:", LANDING_PATH)


data_date,status,file_count,distinct_files
2026-10-07,SUCCESS,1,1
2026-10-08,SUCCESS,1,1


Total Products manifest records: 2
Products landing path: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/


In [0]:

# ============================================================
# CELL 9 — DISCOVER PRODUCTS LANDING FILES
# Read-only
# ============================================================

landing_entries = dbutils.fs.ls(LANDING_PATH)

landing_files = [
    {
        "file_name": entry.name.rstrip("/"),
        "file_path": entry.path,
        "file_size_bytes": entry.size
    }
    for entry in landing_entries
    if not entry.isDir()
]

landing_files_df = spark.createDataFrame(
    landing_files,
    schema="""
        file_name STRING,
        file_path STRING,
        file_size_bytes LONG
    """
) if landing_files else spark.createDataFrame(
    [],
    schema="""
        file_name STRING,
        file_path STRING,
        file_size_bytes LONG
    """
)

print("Products landing files:", len(landing_files))
display(landing_files_df.orderBy("file_name"))


Products landing files: 0


file_name,file_path,file_size_bytes


In [0]:

# ============================================================
# CELL 10 — INSPECT PRODUCTS STORAGE DIRECTORIES
# Read-only
# ============================================================

paths_to_check = [
    BASE_PATH,
    BASE_PATH + "landing/",
    LANDING_PATH,
    BASE_PATH + "bronze/",
    BRONZE_PATH
]

for path in paths_to_check:
    print(f"\nPath: {path}")

    try:
        entries = dbutils.fs.ls(path)

        if not entries:
            print("Directory exists but is empty.")
        else:
            for entry in entries[:20]:
                print(
                    f"  {entry.name} | "
                    f"Directory: {entry.isDir()} | "
                    f"Size: {entry.size}"
                )

    except Exception as e:
        print(f"Could not list path: {str(e)[:300]}")



Path: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/
  bronze/ | Directory: True | Size: 0
  checkpoints/ | Directory: True | Size: 0
  gold/ | Directory: True | Size: 0
  landing/ | Directory: True | Size: 0
  metadata/ | Directory: True | Size: 0
  quarantine/ | Directory: True | Size: 0
  silver/ | Directory: True | Size: 0

Path: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/
  customers/ | Directory: True | Size: 0
  orders/ | Directory: True | Size: 0
  products/ | Directory: True | Size: 0

Path: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/
  year=2026/ | Directory: True | Size: 0

Path: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/
  customers/ | Directory: True | Size: 0
  orders/ | Directory: True | Size: 0
  products/ | Directory: True | Size: 0

Path: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/pr

In [0]:

# ============================================================
# CELL 11 — RECURSIVE PRODUCTS LANDING DISCOVERY
# Read-only; Serverless-compatible
# ============================================================

def list_files_recursive(directory_path):
    discovered_files = []

    for entry in dbutils.fs.ls(directory_path):
        if entry.isDir():
            discovered_files.extend(
                list_files_recursive(entry.path)
            )
        else:
            discovered_files.append({
                "file_name": entry.name,
                "file_path": entry.path,
                "file_size_bytes": entry.size
            })

    return discovered_files


landing_files = list_files_recursive(LANDING_PATH)

if landing_files:
    landing_files_df = spark.createDataFrame(landing_files)
else:
    landing_files_df = spark.createDataFrame(
        [],
        """
        file_name STRING,
        file_path STRING,
        file_size_bytes LONG
        """
    )

print("Products landing files discovered:", len(landing_files))

display(
    landing_files_df.orderBy("file_path")
)


Products landing files discovered: 2


file_name,file_path,file_size_bytes
products_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=07/products_20261007.json,3662500
products_20261008.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=08/products_20261008.json,2615396


In [0]:

# ============================================================
# CELL 12 — COMPARE LANDING FILES WITH MANIFEST
# Read-only; Serverless-compatible
# ============================================================

from pyspark.sql import functions as F

manifest_df = spark.read.format("delta").load(MANIFEST_PATH)

# Files discovered recursively in landing
discovered_df = (
    landing_files_df
    .withColumn("dataset", F.lit(DATASET_NAME))
)

# Distinct registered filenames for Products
registered_df = (
    manifest_df
    .filter(F.col("dataset") == DATASET_NAME)
    .select("file_name", "status", "content_hash", "data_date")
    .dropDuplicates(["file_name"])
)

# Identify registered and unregistered files
discovery_comparison_df = (
    discovered_df.alias("d")
    .join(
        registered_df.alias("m"),
        on="file_name",
        how="left"
    )
    .select(
        F.col("d.file_name"),
        F.col("d.file_path"),
        F.col("d.file_size_bytes"),
        F.col("m.status").alias("manifest_status"),
        F.col("m.content_hash").alias("manifest_content_hash"),
        F.col("m.data_date").alias("manifest_data_date")
    )
    .withColumn(
        "discovery_status",
        F.when(
            F.col("manifest_status").isNull(),
            F.lit("UNREGISTERED")
        ).otherwise(F.lit("REGISTERED"))
    )
    .orderBy("file_name")
)

display(discovery_comparison_df)

print("Discovered landing files:", landing_files_df.count())
print("Unregistered files:", discovery_comparison_df.filter(
    F.col("discovery_status") == "UNREGISTERED"
).count())


file_name,file_path,file_size_bytes,manifest_status,manifest_content_hash,manifest_data_date,discovery_status
products_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=07/products_20261007.json,3662500,SUCCESS,70cef0bfef9126bf3502c9c40139e0f5158f0427ee440ac8cee0dfb43f083118,2026-10-07,REGISTERED
products_20261008.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=08/products_20261008.json,2615396,SUCCESS,647d55b2a4e1b6a7c360248682256d5bca7c0ac36d5ce5be1d0cbaab028c0b4b,2026-10-08,REGISTERED


Discovered landing files: 2
Unregistered files: 0


In [0]:

# ============================================================
# CELL 13 — VERIFY LANDING CONTENT HASHES
# Read-only; Serverless-compatible
# ============================================================

from pyspark.sql import functions as F

# Calculate SHA-256 from the complete binary file contents
landing_hashes_df = (
    spark.read
    .format("binaryFile")
    .load(LANDING_PATH + "**")
    .select(
        F.element_at(
            F.split(F.col("path"), "/"),
            -1
        ).alias("file_name"),
        F.col("path").alias("file_path"),
        F.col("length").alias("file_size_bytes"),
        F.sha2(F.col("content"), 256).alias("actual_content_hash")
    )
)

# Compare actual hashes against the registered manifest hashes
hash_comparison_df = (
    discovery_comparison_df.alias("d")
    .join(
        landing_hashes_df.alias("h"),
        on="file_path",
        how="inner"
    )
    .select(
        F.col("d.file_name"),
        F.col("d.manifest_status"),
        F.col("d.manifest_content_hash"),
        F.col("h.actual_content_hash")
    )
    .withColumn(
        "hash_status",
        F.when(
            F.col("manifest_content_hash").isNull(),
            F.lit("MISSING_MANIFEST_HASH")
        ).when(
            F.col("manifest_content_hash")
            == F.col("actual_content_hash"),
            F.lit("MATCH")
        ).otherwise(F.lit("CONTENT_CHANGED"))
    )
)

display(hash_comparison_df)


file_name,manifest_status,manifest_content_hash,actual_content_hash,hash_status
products_20261007.json,SUCCESS,70cef0bfef9126bf3502c9c40139e0f5158f0427ee440ac8cee0dfb43f083118,70cef0bfef9126bf3502c9c40139e0f5158f0427ee440ac8cee0dfb43f083118,MATCH
products_20261008.json,SUCCESS,647d55b2a4e1b6a7c360248682256d5bca7c0ac36d5ce5be1d0cbaab028c0b4b,647d55b2a4e1b6a7c360248682256d5bca7c0ac36d5ce5be1d0cbaab028c0b4b,MATCH


In [0]:

from pyspark.sql import functions as F

# 1. Landing files with verified actual hashes
landing_candidates_df = (
    discovery_comparison_df.alias("d")
    .join(
        hash_comparison_df.select(
            "file_name",
            "actual_content_hash",
            "hash_status"
        ).alias("h"),
        on="file_name",
        how="left"
    )
    .withColumn("dataset", F.lit(DATASET_NAME))
    .filter(F.col("hash_status") == "MATCH")
)

# 2. Read registered manifest records
registered_manifest_df = (
    spark.read.format("delta").load(MANIFEST_PATH)
    .filter(F.col("dataset") == DATASET_NAME)
    .select(
        "dataset",
        "file_name",
        "data_date",
        "status",
        "content_hash",
        "retry_count"
    )
    .dropDuplicates(["dataset", "file_name"])
)

# 3. Join landing files to manifest records
candidates_df = (
    landing_candidates_df.alias("l")
    .join(
        registered_manifest_df.alias("m"),
        on=["dataset", "file_name"],
        how="left"
    )
    .select(
        F.col("dataset"),
        F.col("file_name"),
        F.col("l.file_path").alias("landing_path"),
        F.col("l.file_size_bytes"),
        F.col("l.actual_content_hash"),
        F.col("l.hash_status"),
        F.col("m.status").alias("manifest_status"),
        F.col("m.data_date").alias("manifest_data_date"),
        F.col("m.content_hash").alias("manifest_content_hash"),
        F.col("m.retry_count")
    )
    .withColumn(
        "selection_reason",
        F.when(
            F.col("manifest_status").isNull(),
            "REGISTER_REQUIRED"
        ).when(
            F.col("actual_content_hash")
            != F.col("manifest_content_hash"),
            "CONTENT_CHANGED"
        ).when(
            F.col("manifest_status") == "PROCESSING",
            "SKIP_PROCESSING"
        ).when(
            F.col("manifest_status").isin("NEW", "FAILED"),
            "ELIGIBLE"
        ).when(
            (F.col("manifest_status") == "SUCCESS")
            & F.lit(REPROCESS_SUCCESSFUL),
            "ELIGIBLE_REPROCESS"
        ).when(
            F.col("manifest_status") == "SUCCESS",
            "SKIP_SUCCESS"
        ).otherwise("REVIEW_STATUS")
    )
)

# 4. Apply run-mode selection
if RUN_MODE == "INCREMENTAL":
    selected_candidates_df = candidates_df.filter(
        F.col("selection_reason").isin(
            "ELIGIBLE", "CONTENT_CHANGED"
        )
    )

elif RUN_MODE == "BACKFILL":
    selected_candidates_df = candidates_df.filter(
        F.to_date("manifest_data_date").between(
            F.lit(BACKFILL_START_DATE).cast("date"),
            F.lit(BACKFILL_END_DATE).cast("date")
        )
        & F.col("selection_reason").isin(
            "ELIGIBLE",
            "CONTENT_CHANGED",
            "ELIGIBLE_REPROCESS"
        )
    )

elif RUN_MODE == "FULL_LOAD":
    selected_candidates_df = candidates_df.filter(
        F.col("selection_reason").isin(
            "REGISTER_REQUIRED",
            "ELIGIBLE",
            "CONTENT_CHANGED",
            "ELIGIBLE_REPROCESS"
        )
    )

display(
    candidates_df.select(
        "file_name",
        "manifest_status",
        "hash_status",
        "selection_reason"
    ).orderBy("file_name")
)

print("Run mode:", RUN_MODE)
print("Candidate files selected:", selected_candidates_df.count())


file_name,manifest_status,hash_status,selection_reason
products_20261007.json,SUCCESS,MATCH,SKIP_SUCCESS
products_20261008.json,SUCCESS,MATCH,SKIP_SUCCESS


Run mode: INCREMENTAL
Candidate files selected: 0


In [0]:

# ============================================================
# CELL 15 — RUN-MODE SELECTION PREVIEW
# Read-only; does not change RUN_MODE or write any data
# ============================================================

from pyspark.sql import functions as F

def preview_run_mode(mode, start_date=None, end_date=None,
                     reprocess_successful=False):

    eligible = candidates_df.filter(
        F.col("manifest_status").isNull()
        | F.col("selection_reason").isin(
            "ELIGIBLE", "CONTENT_CHANGED"
        )
        | (
            F.lit(reprocess_successful)
            & (F.col("selection_reason") == "SKIP_SUCCESS")
        )
    )

    if mode == "INCREMENTAL":
        pass

    elif mode == "BACKFILL":
        if not start_date or not end_date:
            print("BACKFILL: enter start and end dates to preview.")
            return

        eligible = eligible.filter(
            F.to_date("manifest_data_date").between(
                F.lit(start_date).cast("date"),
                F.lit(end_date).cast("date")
            )
        )

    elif mode == "FULL_LOAD":
        pass

    print(
        f"{mode}: {eligible.count()} candidate(s) "
        f"(reprocess_successful={reprocess_successful})"
    )

preview_run_mode("INCREMENTAL")
preview_run_mode(
    "BACKFILL",
    BACKFILL_START_DATE,
    BACKFILL_END_DATE
)
preview_run_mode("FULL_LOAD")
preview_run_mode("FULL_LOAD", reprocess_successful=True)


INCREMENTAL: 0 candidate(s) (reprocess_successful=False)
BACKFILL: enter start and end dates to preview.
FULL_LOAD: 0 candidate(s) (reprocess_successful=False)
FULL_LOAD: 2 candidate(s) (reprocess_successful=True)


In [0]:

# ============================================================
# CELL 16 — BACKFILL DATE-RANGE TEST
# Read-only
# ============================================================

from pyspark.sql import functions as F

BACKFILL_TEST_START = "2026-10-07"
BACKFILL_TEST_END = "2026-10-07"

backfill_test_df = candidates_df.filter(
    F.to_date("manifest_data_date").between(
        F.lit(BACKFILL_TEST_START).cast("date"),
        F.lit(BACKFILL_TEST_END).cast("date")
    )
)

display(
    backfill_test_df.select(
        "file_name",
        "manifest_data_date",
        "manifest_status",
        "selection_reason"
    ).orderBy("file_name")
)

print("Files in date range:", backfill_test_df.count())
print(
    "Eligible without reprocessing:",
    backfill_test_df.filter(
        F.col("selection_reason").isin("ELIGIBLE", "CONTENT_CHANGED")
    ).count()
)
print(
    "Eligible with successful-file reprocessing:",
    backfill_test_df.filter(
        F.col("selection_reason").isin(
            "ELIGIBLE", "CONTENT_CHANGED", "SKIP_SUCCESS"
        )
    ).count()
)


file_name,manifest_data_date,manifest_status,selection_reason
products_20261007.json,2026-10-07,SUCCESS,SKIP_SUCCESS


Files in date range: 1
Eligible without reprocessing: 0
Eligible with successful-file reprocessing: 1


In [0]:

# ============================================================
# CELL 17 — DERIVE AND VALIDATE FILE DATES
# Read-only
# ============================================================

from pyspark.sql import functions as F

dated_candidates_df = (
    candidates_df
    .withColumn(
        "filename_date",
        F.to_date(
            F.regexp_extract(
                F.col("file_name"),
                r"_(\d{8})\.json$",
                1
            ),
            "yyyyMMdd"
        )
    )
    .withColumn(
        "effective_data_date",
        F.coalesce(
            F.to_date("manifest_data_date"),
            F.col("filename_date")
        )
    )
    .withColumn(
        "date_validation",
        F.when(
            F.col("filename_date").isNull(),
            "INVALID_FILENAME_DATE"
        ).when(
            F.col("manifest_data_date").isNotNull()
            & (
                F.to_date("manifest_data_date")
                != F.col("filename_date")
            ),
            "DATE_MISMATCH"
        ).otherwise("VALID")
    )
)

display(
    dated_candidates_df.select(
        "file_name",
        "manifest_data_date",
        "filename_date",
        "effective_data_date",
        "date_validation",
        "selection_reason"
    ).orderBy("file_name")
)

print(
    "Invalid or mismatched dates:",
    dated_candidates_df.filter(
        F.col("date_validation") != "VALID"
    ).count()
)


file_name,manifest_data_date,filename_date,effective_data_date,date_validation,selection_reason
products_20261007.json,2026-10-07,2026-10-07,2026-10-07,VALID,SKIP_SUCCESS
products_20261008.json,2026-10-08,2026-10-08,2026-10-08,VALID,SKIP_SUCCESS


Invalid or mismatched dates: 0


In [0]:

# ============================================================
# CELL 18 — FINAL RUN-MODE SELECTION
# Read-only; no manifest or Bronze writes
# ============================================================

from pyspark.sql import functions as F

# Start from date-validated landing candidates
selector_df = dated_candidates_df.filter(
    F.col("date_validation") == "VALID"
)

# Identify files that are safe to consider
selector_df = selector_df.withColumn(
    "selection_reason",
    F.when(
        F.col("manifest_status").isNull(),
        "REGISTER_REQUIRED"
    ).when(
        F.col("manifest_status") == "PROCESSING",
        "SKIP_PROCESSING"
    ).when(
        F.col("actual_content_hash")
        != F.col("manifest_content_hash"),
        "CONTENT_CHANGED"
    ).when(
        F.col("manifest_status").isin("NEW", "FAILED"),
        "ELIGIBLE"
    ).when(
        (F.col("manifest_status") == "SUCCESS")
        & F.lit(REPROCESS_SUCCESSFUL),
        "ELIGIBLE_REPROCESS"
    ).when(
        F.col("manifest_status") == "SUCCESS",
        "SKIP_SUCCESS"
    ).otherwise("REVIEW_STATUS")
)

# Apply the requested run mode
if RUN_MODE == "INCREMENTAL":
    selected_candidates_df = selector_df.filter(
        F.col("selection_reason").isin(
            "REGISTER_REQUIRED",
            "ELIGIBLE",
            "CONTENT_CHANGED"
        )
    )

elif RUN_MODE == "BACKFILL":
    selected_candidates_df = selector_df.filter(
        F.col("effective_data_date").between(
            F.lit(BACKFILL_START_DATE).cast("date"),
            F.lit(BACKFILL_END_DATE).cast("date")
        )
        & F.col("selection_reason").isin(
            "REGISTER_REQUIRED",
            "ELIGIBLE",
            "CONTENT_CHANGED",
            "ELIGIBLE_REPROCESS"
        )
    )

elif RUN_MODE == "FULL_LOAD":
    selected_candidates_df = selector_df.filter(
        F.col("selection_reason").isin(
            "REGISTER_REQUIRED",
            "ELIGIBLE",
            "CONTENT_CHANGED",
            "ELIGIBLE_REPROCESS"
        )
    )

display(
    selector_df.select(
        "file_name",
        "effective_data_date",
        "manifest_status",
        "hash_status",
        "selection_reason"
    ).orderBy("effective_data_date", "file_name")
)

print("Run mode:", RUN_MODE)
print("Reprocess successful:", REPROCESS_SUCCESSFUL)
print("Selected files:", selected_candidates_df.count())


file_name,effective_data_date,manifest_status,hash_status,selection_reason
products_20261007.json,2026-10-07,SUCCESS,MATCH,SKIP_SUCCESS
products_20261008.json,2026-10-08,SUCCESS,MATCH,SKIP_SUCCESS


Run mode: INCREMENTAL
Reprocess successful: False
Selected files: 0


In [0]:

# ============================================================
# CELL 19 — FINAL SELECTION SAFETY CHECKS
# Read-only
# ============================================================

from pyspark.sql import functions as F

duplicate_count = (
    selected_candidates_df
    .groupBy("dataset", "file_name")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_date_count = selected_candidates_df.filter(
    F.col("date_validation") != "VALID"
).count()

processing_count = selected_candidates_df.filter(
    F.col("manifest_status") == "PROCESSING"
).count()

print("Selected files:", selected_candidates_df.count())
print("Duplicate selections:", duplicate_count)
print("Invalid dates:", invalid_date_count)
print("Already processing:", processing_count)

assert duplicate_count == 0, "Duplicate file selections found"
assert invalid_date_count == 0, "Invalid file dates found"
assert processing_count == 0, "PROCESSING files must not be selected"

print("PASS: Final selection safety checks")


Selected files: 0
Duplicate selections: 0
Invalid dates: 0
Already processing: 0
PASS: Final selection safety checks


In [0]:

# ============================================================
# CELL 20 — PREPARE MANIFEST REGISTRATION CANDIDATES
# Read-only
# ============================================================

from pyspark.sql import functions as F

registration_candidates_df = (
    selector_df
    .filter(
        (F.col("selection_reason") == "REGISTER_REQUIRED")
        & (F.col("date_validation") == "VALID")
        & (F.col("hash_status") == "MATCH")
    )
    .select(
        "dataset",
        "file_name",
        F.col("landing_path").alias("file_path"),
        F.col("effective_data_date").alias("data_date"),
        F.col("file_size_bytes"),
        F.col("actual_content_hash").alias("content_hash")
    )
    .dropDuplicates(["dataset", "file_name"])
)

display(registration_candidates_df)

print(
    "Files requiring registration:",
    registration_candidates_df.count()
)


dataset,file_name,file_path,data_date,file_size_bytes,content_hash


Files requiring registration: 0


In [0]:

# ============================================================
# CELL 21 — REGISTRATION SAFETY VALIDATION
# Read-only
# ============================================================

from pyspark.sql import functions as F

registration_count = registration_candidates_df.count()

duplicate_registration_count = (
    registration_candidates_df
    .groupBy("dataset", "file_name")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_registration_count = (
    registration_candidates_df
    .filter(
        F.col("dataset").isNull()
        | F.col("file_name").isNull()
        | F.col("file_path").isNull()
        | F.col("data_date").isNull()
        | F.col("file_size_bytes").isNull()
        | F.col("content_hash").isNull()
        | (F.col("file_size_bytes") <= 0)
        | (F.length("content_hash") != 64)
    )
    .count()
)

print("Registration candidates:", registration_count)
print("Duplicate registration keys:", duplicate_registration_count)
print("Invalid registration records:", invalid_registration_count)

assert duplicate_registration_count == 0, "Duplicate registration keys found"
assert invalid_registration_count == 0, "Invalid registration metadata found"

print("PASS: Registration candidates are valid")


Registration candidates: 0
Duplicate registration keys: 0
Invalid registration records: 0
PASS: Registration candidates are valid


In [0]:

# ============================================================
# CELL 22 — INSPECT MANIFEST SCHEMA
# Read-only; no writes
# ============================================================

manifest_df = spark.read.format("delta").load(MANIFEST_PATH)

manifest_schema_df = spark.createDataFrame(
    [
        (
            field.name,
            field.dataType.simpleString(),
            field.nullable
        )
        for field in manifest_df.schema.fields
    ],
    ["column_name", "data_type", "nullable"]
)

source_columns = set(registration_source_df.columns)

manifest_schema_df = manifest_schema_df.withColumn(
    "in_registration_source",
    F.col("column_name").isin(list(source_columns))
)

display(
    manifest_schema_df.orderBy(
        F.col("in_registration_source").asc(),
        "column_name"
    )
)

print("Manifest column count:", len(manifest_df.columns))
print("Registration source column count:", len(registration_source_df.columns))
print(
    "Missing columns:",
    sorted(set(manifest_df.columns) - source_columns)
)


column_name,data_type,nullable,in_registration_source
batch_id,string,true,false
claimed_at,timestamp,true,false
discovery_run_id,string,true,false
error_message,string,true,false
error_type,string,true,false
last_attempt_at,timestamp,true,false
last_error_at,timestamp,true,false
last_success_at,timestamp,true,false
lease_expires_at,timestamp,true,false
processed_at,timestamp,true,false


Manifest column count: 32
Registration source column count: 12
Missing columns: ['batch_id', 'claimed_at', 'discovery_run_id', 'error_message', 'error_type', 'last_attempt_at', 'last_error_at', 'last_success_at', 'lease_expires_at', 'processed_at', 'processed_content_hash', 'processing_duration_seconds', 'rejected_record_count', 'run_id', 'schema_fingerprint', 'source_modified_at', 'source_record_count', 'target_delta_version', 'target_path', 'written_record_count']


In [0]:

# ============================================================
# CELL 23 — BUILD SCHEMA-COMPATIBLE REGISTRATION SOURCE
# Read-only; no manifest writes
# ============================================================

from pyspark.sql import functions as F

manifest_df = spark.read.format("delta").load(MANIFEST_PATH)
manifest_columns = manifest_df.columns

# Start with the registration fields already prepared in Cell 22
registration_source_df = registration_candidates_df

# Add registration defaults
registration_source_df = (
    registration_source_df
    .withColumn("status", F.lit("NEW"))
    .withColumn("first_seen", F.current_timestamp())
    .withColumn("last_seen_at", F.current_timestamp())
    .withColumn("status_updated_at", F.current_timestamp())
    .withColumn("retry_count", F.lit(0).cast("int"))
    .withColumn("attempt_count", F.lit(0).cast("int"))
)

# Populate remaining manifest columns with typed NULLs.
# These fields are nullable according to the inspected schema.
for field in manifest_df.schema.fields:
    if field.name not in registration_source_df.columns:
        registration_source_df = registration_source_df.withColumn(
            field.name,
            F.lit(None).cast(field.dataType)
        )

# Match the exact manifest column order and data types
registration_source_df = registration_source_df.select(
    *manifest_columns
)

print("Manifest columns:", len(manifest_columns))
print("Registration source columns:", len(registration_source_df.columns))
print(
    "Schema column mismatch:",
    set(manifest_columns) != set(registration_source_df.columns)
)

display(registration_source_df)


Manifest columns: 32
Registration source columns: 32
Schema column mismatch: False


dataset,file_name,file_path,batch_id,status,first_seen,processed_at,run_id,retry_count,error_message,data_date,file_size_bytes,source_modified_at,content_hash,last_seen_at,discovery_run_id,claimed_at,lease_expires_at,last_attempt_at,attempt_count,last_success_at,status_updated_at,source_record_count,written_record_count,rejected_record_count,processed_content_hash,schema_fingerprint,target_path,target_delta_version,processing_duration_seconds,error_type,last_error_at


In [0]:

# ============================================================
# CELL 24 — VERIFY MANIFEST STATE BEFORE REGISTRATION
# Read-only
# ============================================================

manifest_check_df = (
    spark.read.format("delta").load(MANIFEST_PATH)
    .filter(F.col("dataset") == DATASET_NAME)
)

display(
    manifest_check_df
    .groupBy("status")
    .count()
    .orderBy("status")
)

print(
    "Products manifest records:",
    manifest_check_df.count()
)

print(
    "Registration candidates:",
    registration_source_df.count()
)

print(
    "Existing PROCESSING records:",
    manifest_check_df.filter(
        F.col("status") == "PROCESSING"
    ).count()
)


status,count
SUCCESS,2


Products manifest records: 2
Registration candidates: 0
Existing PROCESSING records: 0


In [0]:

# ============================================================
# CELL 25 — CORRECTED MANIFEST REGISTRATION MERGE
# Inserts new keys only; matched records remain unchanged
# ============================================================

from delta.tables import DeltaTable

# Validate that the source has the complete manifest schema
manifest_df = spark.read.format("delta").load(MANIFEST_PATH)

assert set(registration_source_df.columns) == set(manifest_df.columns), (
    "Registration source schema does not match the manifest"
)

# Ensure source keys are unique
duplicate_keys = (
    registration_source_df
    .groupBy("dataset", "file_name")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicate_keys == 0, "Duplicate registration keys found"

# Explicitly map every manifest column
insert_mapping = {
    column: F.col(f"source.`{column}`")
    for column in manifest_df.columns
}

manifest_delta = DeltaTable.forPath(spark, MANIFEST_PATH)

(
    manifest_delta.alias("target")
    .merge(
        registration_source_df.alias("source"),
        """
        target.dataset = source.dataset
        AND target.file_name = source.file_name
        """
    )
    .whenNotMatchedInsert(values=insert_mapping)
    .execute()
)

print("Manifest registration MERGE completed.")
print(
    "Registration source rows:",
    registration_source_df.count()
)

# Verify final Products manifest state
final_manifest_df = (
    spark.read.format("delta").load(MANIFEST_PATH)
    .filter(F.col("dataset") == DATASET_NAME)
)

display(
    final_manifest_df
    .groupBy("status")
    .count()
    .orderBy("status")
)

print("Final Products manifest records:", final_manifest_df.count())


Manifest registration MERGE completed.
Registration source rows: 0


status,count
SUCCESS,2


Final Products manifest records: 2


In [0]:

# ============================================================
# CELL 26 — INSPECT CLAIM AND LEASE STATE
# Read-only; no status changes
# ============================================================

from pyspark.sql import functions as F

manifest_df = spark.read.format("delta").load(MANIFEST_PATH)

display(
    manifest_df
    .select(
        "dataset",
        "file_name",
        "status",
        "run_id",
        "claimed_at",
        "lease_expires_at",
        "last_attempt_at",
        "attempt_count",
        "retry_count"
    )
    .filter(
        (F.col("dataset") == DATASET_NAME)
        | (F.col("status") == "PROCESSING")
    )
    .orderBy("dataset", "file_name")
)


dataset,file_name,status,run_id,claimed_at,lease_expires_at,last_attempt_at,attempt_count,retry_count
orders,orders_20261008.json,PROCESSING,eebf67b1-6ce3-4250-bc99-e6fca60f341d,2026-10-11T11:37:52.030Z,2026-10-11T12:37:52.030Z,2026-10-11T11:37:52.030Z,3,1
products,products_20261007.json,SUCCESS,eebf67b1-6ce3-4250-bc99-e6fca60f341d,2026-10-11T11:37:52.030Z,2026-10-11T12:37:52.030Z,2026-10-11T11:37:52.030Z,2,0
products,products_20261008.json,SUCCESS,eebf67b1-6ce3-4250-bc99-e6fca60f341d,2026-10-11T11:37:52.030Z,2026-10-11T12:37:52.030Z,2026-10-11T11:37:52.030Z,2,0


In [0]:

# ============================================================
# CELL 28 — PRODUCTS CLAIM ELIGIBILITY CHECK
# Read-only; no manifest changes
# ============================================================

from pyspark.sql import functions as F

products_manifest_df = (
    spark.read.format("delta").load(MANIFEST_PATH)
    .filter(F.col("dataset") == DATASET_NAME)
)

products_processing_df = products_manifest_df.filter(
    F.col("status") == "PROCESSING"
)

selected_count = selected_candidates_df.count()
processing_count = products_processing_df.count()

print("Products selected for processing:", selected_count)
print("Products files already PROCESSING:", processing_count)

display(
    products_manifest_df.select(
        "file_name",
        "status",
        "run_id",
        "lease_expires_at",
        "attempt_count",
        "retry_count"
    ).orderBy("file_name")
)

assert processing_count == 0, (
    "A Products file is already PROCESSING; inspect its lease before claiming"
)

print("PASS: No Products file is currently PROCESSING")


Products selected for processing: 0
Products files already PROCESSING: 0


file_name,status,run_id,lease_expires_at,attempt_count,retry_count
products_20261007.json,SUCCESS,eebf67b1-6ce3-4250-bc99-e6fca60f341d,2026-10-11T12:37:52.030Z,2,0
products_20261008.json,SUCCESS,eebf67b1-6ce3-4250-bc99-e6fca60f341d,2026-10-11T12:37:52.030Z,2,0


PASS: No Products file is currently PROCESSING


In [0]:

# ============================================================
# CELL 29 — PREPARE PRODUCTS CLAIM CANDIDATES
# Read-only; no manifest changes
# ============================================================

from pyspark.sql import functions as F
import uuid

# Create a unique run ID for this execution
EXECUTION_RUN_ID = str(uuid.uuid4())

# Only claim candidates selected by the validated selector
claim_candidates_df = (
    selected_candidates_df
    .filter(F.col("dataset") == DATASET_NAME)
    .filter(
        F.col("selection_reason").isin(
            "REGISTER_REQUIRED",
            "ELIGIBLE",
            "CONTENT_CHANGED",
            "ELIGIBLE_REPROCESS"
        )
    )
    .select(
        "dataset",
        "file_name",
        "landing_path",
        "actual_content_hash",
        "effective_data_date",
        "selection_reason"
    )
    .dropDuplicates(["dataset", "file_name"])
)

print("Execution run ID:", EXECUTION_RUN_ID)
print("Dataset:", DATASET_NAME)
print("Claim candidates:", claim_candidates_df.count())

display(claim_candidates_df)


Execution run ID: 01f427be-119c-424e-951a-d9037cc441b4
Dataset: products
Claim candidates: 0


dataset,file_name,landing_path,actual_content_hash,effective_data_date,selection_reason


In [0]:

# ============================================================
# CELL 30 — CLAIM CANDIDATE SAFETY CHECK
# Read-only
# ============================================================

from pyspark.sql import functions as F

claim_candidate_count = claim_candidates_df.count()

duplicate_claim_keys = (
    claim_candidates_df
    .groupBy("dataset", "file_name")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

wrong_dataset_count = claim_candidates_df.filter(
    F.col("dataset") != DATASET_NAME
).count()

invalid_claim_count = claim_candidates_df.filter(
    F.col("file_name").isNull()
    | F.col("landing_path").isNull()
    | F.col("actual_content_hash").isNull()
    | F.col("effective_data_date").isNull()
).count()

print("Claim candidates:", claim_candidate_count)
print("Duplicate claim keys:", duplicate_claim_keys)
print("Wrong-dataset candidates:", wrong_dataset_count)
print("Invalid claim candidates:", invalid_claim_count)

assert duplicate_claim_keys == 0, "Duplicate claim keys found"
assert wrong_dataset_count == 0, "Wrong dataset in claim candidates"
assert invalid_claim_count == 0, "Invalid claim candidates found"

print("PASS: Claim candidate safety checks")


Claim candidates: 0
Duplicate claim keys: 0
Wrong-dataset candidates: 0
Invalid claim candidates: 0
PASS: Claim candidate safety checks


In [0]:

# ============================================================
# CELL 31 — CLAIM PLAN PREVIEW
# Read-only; no manifest writes
# ============================================================

from pyspark.sql import functions as F

claim_plan_df = (
    claim_candidates_df
    .withColumn("claim_run_id", F.lit(EXECUTION_RUN_ID))
    .withColumn("proposed_status", F.lit("PROCESSING"))
    .withColumn("proposed_claimed_at", F.current_timestamp())
    .withColumn(
        "proposed_lease_expires_at",
        F.expr("current_timestamp() + INTERVAL 60 MINUTES")
    )
)

display(claim_plan_df)

print("Execution run ID:", EXECUTION_RUN_ID)
print("Files planned for claim:", claim_plan_df.count())
print("Manifest writes performed: 0")


dataset,file_name,landing_path,actual_content_hash,effective_data_date,selection_reason,claim_run_id,proposed_status,proposed_claimed_at,proposed_lease_expires_at


Execution run ID: 01f427be-119c-424e-951a-d9037cc441b4
Files planned for claim: 0
Manifest writes performed: 0


In [0]:

# ============================================================
# CELL 32 — CREATE ISOLATED TEST MANIFEST
# Writes only to a dedicated test path
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType,
    TimestampType, IntegerType
)
from datetime import datetime, timezone, timedelta

TEST_MANIFEST_PATH = (
    BASE_PATH + "metadata/test_file_manifest/"
)

test_schema = StructType([
    StructField("dataset", StringType(), False),
    StructField("file_name", StringType(), False),
    StructField("status", StringType(), False),
    StructField("run_id", StringType(), True),
    StructField("claimed_at", TimestampType(), True),
    StructField("lease_expires_at", TimestampType(), True),
    StructField("retry_count", IntegerType(), False),
    StructField("attempt_count", IntegerType(), False),
    StructField("content_hash", StringType(), True),
    StructField("error_message", StringType(), True)
])

now_utc = datetime.now(timezone.utc).replace(tzinfo=None)

test_records = [
    (
        "test_products",
        "test_product_new.json",
        "NEW",
        None, None, None,
        0, 0,
        "test_hash_new",
        None
    ),
    (
        "test_products",
        "test_product_processing.json",
        "PROCESSING",
        "test-existing-run",
        now_utc - timedelta(minutes=90),
        now_utc - timedelta(minutes=30),
        1, 2,
        "test_hash_processing",
        None
    )
]

test_manifest_df = spark.createDataFrame(
    test_records,
    schema=test_schema
)

# Refuse to overwrite an existing test table.

# Write only if the isolated test path does not already exist
if DeltaTable.isDeltaTable(spark, TEST_MANIFEST_PATH):
    raise Exception(
        "Test manifest already exists. "
        "Refusing to overwrite existing test data."
    )

(
    test_manifest_df.write
    .format("delta")
    .mode("error")
    .save(TEST_MANIFEST_PATH)
)


print("Isolated test manifest created.")
print("Test path:", TEST_MANIFEST_PATH)

display(
    spark.read.format("delta")
    .load(TEST_MANIFEST_PATH)
    .select(
        "dataset",
        "file_name",
        "status",
        "run_id",
        "lease_expires_at",
        "attempt_count",
        "retry_count"
    )
)


Isolated test manifest created.
Test path: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/metadata/test_file_manifest/


dataset,file_name,status,run_id,lease_expires_at,attempt_count,retry_count
test_products,test_product_new.json,NEW,null,null,0,0
test_products,test_product_processing.json,PROCESSING,test-existing-run,2026-10-11T12:38:26.530Z,2,1


In [0]:

# ============================================================
# CELL 33 — CLAIM ONE TEST FILE (CORRECTED)
# Writes only to the isolated test manifest
# ============================================================

from delta.tables import DeltaTable
from pyspark.sql import functions as F
import uuid

TEST_DATASET = "test_products"
TEST_FILE = "test_product_new.json"
TEST_RUN_ID = str(uuid.uuid4())
LEASE_MINUTES = 60

test_delta = DeltaTable.forPath(spark, TEST_MANIFEST_PATH)

(
    test_delta.alias("target")
    .update(
        condition=(
            (F.col("target.dataset") == TEST_DATASET)
            & (F.col("target.file_name") == TEST_FILE)
            & (F.col("target.status").isin("NEW", "FAILED"))
        ),
        set={
            "status": F.lit("PROCESSING"),
            "run_id": F.lit(TEST_RUN_ID),
            "claimed_at": F.current_timestamp(),
            "lease_expires_at": F.expr(
                f"current_timestamp() + INTERVAL {LEASE_MINUTES} MINUTES"
            ),
            "attempt_count": F.col("target.attempt_count") + 1
        }
    )
)

claimed_record_df = (
    spark.read.format("delta")
    .load(TEST_MANIFEST_PATH)
    .filter(
        (F.col("dataset") == TEST_DATASET)
        & (F.col("file_name") == TEST_FILE)
    )
)

display(
    claimed_record_df.select(
        "dataset",
        "file_name",
        "status",
        "run_id",
        "claimed_at",
        "lease_expires_at",
        "attempt_count",
        "retry_count"
    )
)

print("Expected run ID:", TEST_RUN_ID)


dataset,file_name,status,run_id,claimed_at,lease_expires_at,attempt_count,retry_count
test_products,test_product_new.json,PROCESSING,dbfc8035-9df2-4896-abca-8e305dc8f313,2026-10-11T13:09:55.963Z,2026-10-11T14:09:55.963Z,1,0


Expected run ID: dbfc8035-9df2-4896-abca-8e305dc8f313


In [0]:

# ============================================================
# CELL 34 — VERIFY TEST CLAIM OWNERSHIP
# Read-only
# ============================================================

from pyspark.sql import functions as F

claimed_record_df = (
    spark.read.format("delta")
    .load(TEST_MANIFEST_PATH)
    .filter(
        (F.col("dataset") == TEST_DATASET)
        & (F.col("file_name") == TEST_FILE)
    )
)

ownership_check_df = claimed_record_df.withColumn(
    "ownership_check",
    F.when(
        (F.col("status") == "PROCESSING")
        & (F.col("run_id") == F.lit(TEST_RUN_ID))
        & (F.col("lease_expires_at") > F.current_timestamp()),
        "PASS"
    ).otherwise("FAIL")
)

display(
    ownership_check_df.select(
        "file_name",
        "status",
        "run_id",
        "lease_expires_at",
        "ownership_check"
    )
)

assert ownership_check_df.filter(
    F.col("ownership_check") != "PASS"
).count() == 0, "Claim ownership validation failed"

print("PASS: Claim ownership and active lease verified")


file_name,status,run_id,lease_expires_at,ownership_check
test_product_new.json,PROCESSING,dbfc8035-9df2-4896-abca-8e305dc8f313,2026-10-11T14:09:55.963Z,PASS


PASS: Claim ownership and active lease verified


In [0]:

# ============================================================
# CELL 35 — VERIFY EXPIRED TEST LEASE
# Read-only
# ============================================================

from pyspark.sql import functions as F

EXPIRED_TEST_FILE = "test_product_processing.json"

expired_test_df = (
    spark.read.format("delta")
    .load(TEST_MANIFEST_PATH)
    .filter(
        (F.col("dataset") == TEST_DATASET)
        & (F.col("file_name") == EXPIRED_TEST_FILE)
    )
    .withColumn(
        "lease_state",
        F.when(
            F.col("lease_expires_at").isNull(),
            "MISSING_EXPIRY"
        ).when(
            F.col("lease_expires_at") <= F.current_timestamp(),
            "EXPIRED"
        ).otherwise("ACTIVE")
    )
)

display(
    expired_test_df.select(
        "file_name",
        "status",
        "run_id",
        "claimed_at",
        "lease_expires_at",
        "attempt_count",
        "retry_count",
        "lease_state"
    )
)

assert expired_test_df.count() == 1, (
    "Expected exactly one expired-lease test record"
)

print("Expired-lease inspection complete.")


file_name,status,run_id,claimed_at,lease_expires_at,attempt_count,retry_count,lease_state
test_product_processing.json,PROCESSING,test-existing-run,2026-10-11T11:38:26.530Z,2026-10-11T12:38:26.530Z,2,1,EXPIRED


Expired-lease inspection complete.


In [0]:

# ============================================================
# CELL 36 — RECOVER EXPIRED TEST LEASE
# Writes only to the isolated test manifest
# ============================================================

from delta.tables import DeltaTable
from pyspark.sql import functions as F

EXPIRED_TEST_FILE = "test_product_processing.json"
EXPECTED_OLD_RUN_ID = "test-existing-run"

test_delta = DeltaTable.forPath(spark, TEST_MANIFEST_PATH)

(
    test_delta.alias("target")
    .update(
        condition=(
            (F.col("target.dataset") == TEST_DATASET)
            & (F.col("target.file_name") == EXPIRED_TEST_FILE)
            & (F.col("target.status") == "PROCESSING")
            & (F.col("target.run_id") == EXPECTED_OLD_RUN_ID)
            & (F.col("target.lease_expires_at") <= F.current_timestamp())
        ),
        set={
            "status": F.lit("NEW"),
            "run_id": F.lit(None).cast("string"),
            "claimed_at": F.lit(None).cast("timestamp"),
            "lease_expires_at": F.lit(None).cast("timestamp")
        }
    )
)

recovered_test_df = (
    spark.read.format("delta")
    .load(TEST_MANIFEST_PATH)
    .filter(
        (F.col("dataset") == TEST_DATASET)
        & (F.col("file_name") == EXPIRED_TEST_FILE)
    )
)

display(
    recovered_test_df.select(
        "file_name",
        "status",
        "run_id",
        "lease_expires_at",
        "attempt_count",
        "retry_count"
    )
)


file_name,status,run_id,lease_expires_at,attempt_count,retry_count
test_product_processing.json,NEW,null,null,2,1


In [0]:

# ============================================================
# CELL 37 — FINAL TEST MANIFEST VERIFICATION
# Read-only
# ============================================================

test_manifest_final_df = (
    spark.read.format("delta")
    .load(TEST_MANIFEST_PATH)
)

display(
    test_manifest_final_df.select(
        "dataset",
        "file_name",
        "status",
        "run_id",
        "lease_expires_at",
        "attempt_count",
        "retry_count"
    ).orderBy("file_name")
)

print("Test manifest records:", test_manifest_final_df.count())
print(
    "Records still PROCESSING:",
    test_manifest_final_df.filter(
        F.col("status") == "PROCESSING"
    ).count()
)

print(
    "Records ready for processing:",
    test_manifest_final_df.filter(
        F.col("status") == "NEW"
    ).count()
)


dataset,file_name,status,run_id,lease_expires_at,attempt_count,retry_count
test_products,test_product_new.json,PROCESSING,dbfc8035-9df2-4896-abca-8e305dc8f313,2026-10-11T14:09:55.963Z,1,0
test_products,test_product_processing.json,NEW,null,null,2,1


Test manifest records: 2
Records still PROCESSING: 1
Records ready for processing: 1


In [0]:

# ============================================================
# CELL 38 — VERIFY REAL PRODUCTS MANIFEST IS UNCHANGED
# Read-only; uses the REAL manifest
# ============================================================

from pyspark.sql import functions as F

real_products_df = (
    spark.read.format("delta").load(MANIFEST_PATH)
    .filter(F.col("dataset") == "products")
)

display(
    real_products_df.select(
        "file_name",
        "status",
        "content_hash",
        "processed_content_hash",
        "source_record_count",
        "written_record_count",
        "target_path",
        "target_delta_version"
    ).orderBy("file_name")
)

print("Real Products records:", real_products_df.count())
print(
    "Non-SUCCESS Products records:",
    real_products_df.filter(F.col("status") != "SUCCESS").count()
)


file_name,status,content_hash,processed_content_hash,source_record_count,written_record_count,target_path,target_delta_version
products_20261007.json,SUCCESS,70cef0bfef9126bf3502c9c40139e0f5158f0427ee440ac8cee0dfb43f083118,70cef0bfef9126bf3502c9c40139e0f5158f0427ee440ac8cee0dfb43f083118,25000,25000,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/products/,0
products_20261008.json,SUCCESS,647d55b2a4e1b6a7c360248682256d5bca7c0ac36d5ce5be1d0cbaab028c0b4b,647d55b2a4e1b6a7c360248682256d5bca7c0ac36d5ce5be1d0cbaab028c0b4b,25000,25000,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/products/,0


Real Products records: 2
Non-SUCCESS Products records: 0


In [0]:

# ============================================================
# CELL 2 — LOAD AND VALIDATE CLAIMED PRODUCT FILES
# Read-only validation
# ============================================================

from delta.tables import DeltaTable
from pyspark.sql import functions as F

MANIFEST_RUN_ID = dbutils.widgets.get("manifest_run_id").strip()

if not MANIFEST_RUN_ID:
    raise ValueError("Enter the manifest run ID in the widget.")

if not DeltaTable.isDeltaTable(spark, MANIFEST_PATH):
    raise RuntimeError(f"Manifest Delta table not found: {MANIFEST_PATH}")

manifest_df = spark.read.format("delta").load(MANIFEST_PATH)

required_columns = {
    "dataset", "file_name", "file_path", "batch_id",
    "status", "run_id", "content_hash", "lease_expires_at"
}

missing_columns = required_columns - set(manifest_df.columns)

if missing_columns:
    raise ValueError(f"Missing manifest columns: {sorted(missing_columns)}")

claimed_files_df = (
    manifest_df
    .filter(
        (F.col("dataset") == "products")
        & (F.col("status") == "PROCESSING")
        & (F.col("run_id") == MANIFEST_RUN_ID)
    )
    .select(
        "dataset", "file_name", "file_path", "batch_id",
        "run_id", "content_hash", "lease_expires_at"
    )
)

duplicates_df = (
    claimed_files_df
    .groupBy("dataset", "file_path")
    .count()
    .filter(F.col("count") > 1)
)

if duplicates_df.limit(1).count() > 0:
    display(duplicates_df)
    raise RuntimeError("Duplicate claimed product files found.")

claimed_count = claimed_files_df.count()

print("Dataset:", DATASET_NAME)
print("Manifest run ID:", MANIFEST_RUN_ID)
print("Claimed product files:", claimed_count)

if claimed_count == 0:
    display(
        manifest_df
        .filter(F.col("dataset") == "products")
        .select("file_name", "status", "run_id", "lease_expires_at")
        .orderBy("file_name")
    )
    raise RuntimeError("No Products files are claimed by this run.")

display(claimed_files_df.orderBy("file_name"))


---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
File <command-5499420920101055>, line 13
     10 MANIFEST_RUN_ID = dbutils.widgets.get("manifest_run_id").strip()
     12 if not MANIFEST_RUN_ID:
---> 13     raise ValueError("Enter the manifest run ID in the widget.")
     15 if not DeltaTable.isDeltaTable(spark, MANIFEST_PATH):
     16     raise RuntimeError(f"Manifest Delta table not found: {MANIFEST_PATH}")

ValueError: Enter the manifest run ID in the widget.

In [0]:

# ============================================================
# CELL 3 — VALIDATE PRODUCT CLAIM LEASES
# Read-only validation
# ============================================================

from datetime import datetime, timezone

now_utc = datetime.now(timezone.utc)

lease_records = claimed_files_df.collect()

expired_files = []
valid_files = []

for record in lease_records:
    lease_expiry = record["lease_expires_at"]
    file_name = record["file_name"]

    if lease_expiry is None:
        raise RuntimeError(f"Missing lease expiry: {file_name}")

    if lease_expiry.tzinfo is None:
        lease_expiry = lease_expiry.replace(tzinfo=timezone.utc)

    print(
        f"{file_name} | "
        f"Lease expiry: {lease_expiry} | "
        f"Current UTC: {now_utc}"
    )

    if lease_expiry <= now_utc:
        expired_files.append(file_name)
    else:
        valid_files.append(file_name)

print("\nValid claims:", len(valid_files))
print("Expired claims:", len(expired_files))

if expired_files:
    raise RuntimeError(
        "Expired claims detected: "
        + ", ".join(expired_files)
        + ". Stop and reclaim them through the manifest workflow."
    )

print("All Product claim leases are valid.")


products_20261007.json | Lease expiry: 2026-10-11 12:37:52.030801+00:00 | Current UTC: 2026-10-11 11:57:13.729650+00:00
products_20261008.json | Lease expiry: 2026-10-11 12:37:52.030801+00:00 | Current UTC: 2026-10-11 11:57:13.729650+00:00

Valid claims: 2
Expired claims: 0
All Product claim leases are valid.


In [0]:

# ============================================================
# CELL 4 — INSPECT PRODUCT SOURCE SCHEMA
# Supports multiline JSON arrays
# ============================================================

for record in claimed_files_df.orderBy("file_name").collect():
    file_path = record["file_path"]
    file_name = record["file_name"]

    print(f"\n{'=' * 60}")
    print(f"Inspecting: {file_name}")
    print("=" * 60)

    source_df = (
        spark.read
        .option("multiLine", "true")
        .option("mode", "FAILFAST")
        .json(file_path)
    )

    print("Source schema:")
    source_df.printSchema()

    print("Total source columns:", len(source_df.columns))
    print("Sample records:")
    display(source_df.limit(5))



Inspecting: products_20261007.json
Source schema:
root
 |-- brand: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- product_id: long (nullable = true)
 |-- product_name: string (nullable = true)

Total source columns: 5
Sample records:


brand,category,price,product_id,product_name
BrandA,Beverages,149.99,230001,Product Oct7 Updated 1
BrandB,Snacks,219.49,230002,Product Oct7 Updated 2
BrandC,Nutrition,329.75,230003,Product Oct7 Updated 3
BrandD,Cereals,189.25,230004,Product Oct7 Updated 4
BrandE,Dairy,279.9,230005,Product Oct7 Updated 5



Inspecting: products_20261008.json
Source schema:
root
 |-- brand: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- product_name: string (nullable = true)

Total source columns: 5
Sample records:


brand,category,price,product_id,product_name
Nescafe,Snacks,50,230001,Updated Product 230001
Maggi,Dairy,67,230002,Updated Product 230002
KitKat,Nutrition,84,230003,Updated Product 230003
Gerber,Pet Care,101,230004,Updated Product 230004
Purina,Breakfast,118,230005,Updated Product 230005


In [0]:

# ============================================================
# CELL 5 — DEFINE EXPLICIT PRODUCTS SCHEMA
# ============================================================

from pyspark.sql.types import (
    StructType,
    StructField,
    LongType,
    StringType,
    DoubleType
)

PRODUCTS_SCHEMA = StructType([
    StructField("product_id", LongType(), nullable=False),
    StructField("product_name", StringType(), nullable=True),
    StructField("category", StringType(), nullable=True),
    StructField("brand", StringType(), nullable=True),
    StructField("price", DoubleType(), nullable=True)
])

print("Explicit Products schema:")
PRODUCTS_SCHEMA.simpleString()


Explicit Products schema:


'struct<product_id:bigint,product_name:string,category:string,brand:string,price:double>'

In [0]:

# ============================================================
# CELL 6 — VALIDATE SCHEMA CONSISTENCY
# ============================================================

for record in claimed_files_df.orderBy("file_name").collect():
    file_name = record["file_name"]
    file_path = record["file_path"]

    source_df = (
        spark.read
        .schema(PRODUCTS_SCHEMA)
        .option("multiLine", "true")
        .option("mode", "FAILFAST")
        .json(file_path)
    )

    print(f"\n{'=' * 60}")
    print(f"File: {file_name}")
    print("=" * 60)

    source_df.printSchema()

    print("Column types:", source_df.dtypes)
    print("Record count:", source_df.count())

    display(source_df.limit(5))



File: products_20261007.json
root
 |-- product_id: long (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)

Column types: [('product_id', 'bigint'), ('product_name', 'string'), ('category', 'string'), ('brand', 'string'), ('price', 'double')]
Record count: 25000


product_id,product_name,category,brand,price
230001,Product Oct7 Updated 1,Beverages,BrandA,149.99
230002,Product Oct7 Updated 2,Snacks,BrandB,219.49
230003,Product Oct7 Updated 3,Nutrition,BrandC,329.75
230004,Product Oct7 Updated 4,Cereals,BrandD,189.25
230005,Product Oct7 Updated 5,Dairy,BrandE,279.9



File: products_20261008.json
root
 |-- product_id: long (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)

Column types: [('product_id', 'bigint'), ('product_name', 'string'), ('category', 'string'), ('brand', 'string'), ('price', 'double')]
Record count: 25000


product_id,product_name,category,brand,price
230001,Updated Product 230001,Snacks,Nescafe,50.0
230002,Updated Product 230002,Dairy,Maggi,67.0
230003,Updated Product 230003,Nutrition,KitKat,84.0
230004,Updated Product 230004,Pet Care,Gerber,101.0
230005,Updated Product 230005,Breakfast,Purina,118.0


In [0]:

# ============================================================
# CELL 7 — BRONZE TRANSFORMATION
# ============================================================

from pyspark.sql import functions as F

bronze_dfs = []

for record in claimed_files_df.orderBy("file_name").collect():
    file_name = record["file_name"]
    file_path = record["file_path"]
    batch_id = record["batch_id"]

    print(f"Preparing Bronze data from: {file_name}")

    source_df = (
        spark.read
        .schema(PRODUCTS_SCHEMA)
        .option("multiLine", "true")
        .option("mode", "FAILFAST")
        .json(file_path)
    )

    transformed_df = (
        source_df
        .withColumn("_source_file", F.lit(file_name))
        .withColumn("_source_path", F.lit(file_path))
        .withColumn("_dataset", F.lit(DATASET_NAME))
        .withColumn("_batch_id", F.lit(batch_id))
        .withColumn("_run_id", F.lit(MANIFEST_RUN_ID))
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_system", F.lit(SOURCE_SYSTEM))
    )

    bronze_dfs.append(transformed_df)

if not bronze_dfs:
    raise ValueError("No claimed Products files found.")

bronze_df = bronze_dfs[0]

for next_df in bronze_dfs[1:]:
    bronze_df = bronze_df.unionByName(next_df)

print("Bronze schema:")
bronze_df.printSchema()

print("Total Bronze records:", bronze_df.count())

print("Records by source file:")
display(
    bronze_df.groupBy("_source_file")
    .count()
    .orderBy("_source_file")
)

display(bronze_df.limit(10))


Preparing Bronze data from: products_20261007.json
Preparing Bronze data from: products_20261008.json
Bronze schema:
root
 |-- product_id: long (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- _source_file: string (nullable = false)
 |-- _source_path: string (nullable = false)
 |-- _dataset: string (nullable = false)
 |-- _batch_id: string (nullable = false)
 |-- _run_id: string (nullable = false)
 |-- _ingested_at: timestamp (nullable = false)
 |-- _source_system: string (nullable = false)

Total Bronze records: 50000
Records by source file:


_source_file,count
products_20261007.json,25000
products_20261008.json,25000


product_id,product_name,category,brand,price,_source_file,_source_path,_dataset,_batch_id,_run_id,_ingested_at,_source_system
230001,Product Oct7 Updated 1,Beverages,BrandA,149.99,products_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=07/products_20261007.json,products,9df746d8-611b-493a-adeb-94f07054166a,eebf67b1-6ce3-4250-bc99-e6fca60f341d,2026-10-11T12:03:36.263Z,product_api
230002,Product Oct7 Updated 2,Snacks,BrandB,219.49,products_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=07/products_20261007.json,products,9df746d8-611b-493a-adeb-94f07054166a,eebf67b1-6ce3-4250-bc99-e6fca60f341d,2026-10-11T12:03:36.263Z,product_api
230003,Product Oct7 Updated 3,Nutrition,BrandC,329.75,products_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=07/products_20261007.json,products,9df746d8-611b-493a-adeb-94f07054166a,eebf67b1-6ce3-4250-bc99-e6fca60f341d,2026-10-11T12:03:36.263Z,product_api
230004,Product Oct7 Updated 4,Cereals,BrandD,189.25,products_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=07/products_20261007.json,products,9df746d8-611b-493a-adeb-94f07054166a,eebf67b1-6ce3-4250-bc99-e6fca60f341d,2026-10-11T12:03:36.263Z,product_api
230005,Product Oct7 Updated 5,Dairy,BrandE,279.9,products_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=07/products_20261007.json,products,9df746d8-611b-493a-adeb-94f07054166a,eebf67b1-6ce3-4250-bc99-e6fca60f341d,2026-10-11T12:03:36.263Z,product_api
230006,Product Oct7 Updated 6,Beverages,BrandB,119.5,products_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=07/products_20261007.json,products,9df746d8-611b-493a-adeb-94f07054166a,eebf67b1-6ce3-4250-bc99-e6fca60f341d,2026-10-11T12:03:36.263Z,product_api
260007,Product Oct7 7,Cereals,BrandE,76.59,products_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=07/products_20261007.json,products,9df746d8-611b-493a-adeb-94f07054166a,eebf67b1-6ce3-4250-bc99-e6fca60f341d,2026-10-11T12:03:36.263Z,product_api
260008,Product Oct7 8,Dairy,BrandA,83.96,products_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=07/products_20261007.json,products,9df746d8-611b-493a-adeb-94f07054166a,eebf67b1-6ce3-4250-bc99-e6fca60f341d,2026-10-11T12:03:36.263Z,product_api
260009,Product Oct7 9,Beverages,BrandB,91.33,products_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=07/products_20261007.json,products,9df746d8-611b-493a-adeb-94f07054166a,eebf67b1-6ce3-4250-bc99-e6fca60f341d,2026-10-11T12:03:36.263Z,product_api
260010,Product Oct7 10,Snacks,BrandC,98.7,products_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=07/products_20261007.json,products,9df746d8-611b-493a-adeb-94f07054166a,eebf67b1-6ce3-4250-bc99-e6fca60f341d,2026-10-11T12:03:36.263Z,product_api


In [0]:

# ============================================================
# CELL 8 — BRONZE DATA QUALITY VALIDATION
# ============================================================

from pyspark.sql import functions as F

# 1. Check required fields
invalid_required_df = bronze_df.filter(
    F.col("product_id").isNull()
    | F.col("product_name").isNull()
    | F.col("category").isNull()
    | F.col("brand").isNull()
    | F.col("price").isNull()
)

# 2. Check invalid prices
invalid_price_df = bronze_df.filter(
    F.col("price") < 0
)

# 3. Check duplicate product IDs within each source file
duplicate_ids_df = (
    bronze_df
    .groupBy("_source_file", "product_id")
    .count()
    .filter(F.col("count") > 1)
)

# 4. Display validation results
total_records = bronze_df.count()
invalid_required_count = invalid_required_df.count()
invalid_price_count = invalid_price_df.count()
duplicate_id_groups = duplicate_ids_df.count()

print("========== BRONZE QUALITY REPORT ==========")
print(f"Total records: {total_records}")
print(f"Records with missing required fields: {invalid_required_count}")
print(f"Records with negative prices: {invalid_price_count}")
print(f"Duplicate product-ID groups within files: {duplicate_id_groups}")

print("\nDuplicate ID details:")
display(duplicate_ids_df)

# 5. Stop before writing if any checks fail
if (
    invalid_required_count > 0
    or invalid_price_count > 0
    or duplicate_id_groups > 0
):
    raise ValueError(
        "Bronze data quality validation failed. "
        "Resolve the issues before writing to Delta."
    )

print("\nBRONZE DATA QUALITY VALIDATION PASSED")


========== BRONZE QUALITY REPORT ==========
Total records: 50000
Records with missing required fields: 0
Records with negative prices: 0
Duplicate product-ID groups within files: 0

Duplicate ID details:


_source_file,product_id,count



BRONZE DATA QUALITY VALIDATION PASSED


In [0]:

# ============================================================
# CELL 9 — INSPECT BRONZE TARGET BEFORE WRITING
# ============================================================

from delta.tables import DeltaTable

bronze_target_exists = DeltaTable.isDeltaTable(
    spark,
    BRONZE_PATH
)

print("Bronze target path:", BRONZE_PATH)
print("Existing Delta table:", bronze_target_exists)

if bronze_target_exists:
    existing_bronze_df = spark.read.format("delta").load(BRONZE_PATH)

    print("Existing Bronze record count:", existing_bronze_df.count())

    print("Existing records by source file:")
    display(
        existing_bronze_df
        .groupBy("_source_file", "_run_id")
        .count()
        .orderBy("_source_file", "_run_id")
    )
else:
    print("No existing Delta table found. First write can create it.")


Bronze target path: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/products/
Existing Delta table: False
No existing Delta table found. First write can create it.


In [0]:

# ============================================================
# CELL 10 — CREATE OR SKIP BRONZE DELTA WRITE
# ============================================================

from delta.tables import DeltaTable
from pyspark.sql import functions as F

expected_count = bronze_df.count()

if expected_count == 0:
    raise ValueError("Bronze DataFrame is empty. Write cancelled.")

if DeltaTable.isDeltaTable(spark, BRONZE_PATH):
    print("Bronze Delta table already exists.")
    print("Skipping write to prevent duplicate records.")

else:
    print("Bronze Delta table does not exist.")
    print("Creating table and inserting records...")

    (
        bronze_df.write
        .format("delta")
        .mode("append")
        .save(BRONZE_PATH)
    )

    print("Bronze Delta table created successfully.")

# ============================================================
# VERIFY PERSISTED DATA
# ============================================================

written_df = spark.read.format("delta").load(BRONZE_PATH)
actual_count = written_df.count()

print("\n========== BRONZE WRITE VALIDATION ==========")
print("Expected records for this run:", expected_count)
print("Actual records in Bronze:", actual_count)

# Confirm that the current run's files are present
expected_files = {
    row["file_name"]
    for row in claimed_files_df.select("file_name").collect()
}

actual_files = {
    row["_source_file"]
    for row in (
        written_df
        .filter(F.col("_run_id") == MANIFEST_RUN_ID)
        .select("_source_file")
        .distinct()
        .collect()
    )
}

missing_files = expected_files - actual_files

if missing_files:
    raise RuntimeError(
        f"Bronze verification failed. Missing files: {missing_files}"
    )

if actual_count < expected_count:
    raise RuntimeError(
        "Bronze contains fewer records than expected."
    )

print("All claimed files are present for this run.")
print("BRONZE VERIFICATION PASSED")

print("\nRecords by source file:")
display(
    written_df
    .groupBy("_source_file")
    .count()
    .orderBy("_source_file")
)


Bronze Delta table already exists.
Skipping write to prevent duplicate records.

========== BRONZE WRITE VALIDATION ==========
Expected records for this run: 50000
Actual records in Bronze: 50000
All claimed files are present for this run.
BRONZE VERIFICATION PASSED

Records by source file:


_source_file,count
products_20261007.json,25000
products_20261008.json,25000


In [0]:

# ============================================================
# CELL 10A — VERIFY EXISTING BRONZE DATA
# ============================================================

from delta.tables import DeltaTable
from pyspark.sql import functions as F

if not DeltaTable.isDeltaTable(spark, BRONZE_PATH):
    raise RuntimeError("Bronze Delta table does not exist.")

written_df = spark.read.format("delta").load(BRONZE_PATH)

print("Total Bronze records:", written_df.count())

print("Records by source file:")
display(
    written_df
    .groupBy("_source_file")
    .agg(
        F.count("*").alias("record_count"),
        F.countDistinct("product_id").alias("distinct_product_ids")
    )
    .orderBy("_source_file")
)

print("Records by ingestion run:")
display(
    written_df
    .groupBy("_run_id")
    .count()
    .orderBy("_run_id")
)


Total Bronze records: 50000
Records by source file:


_source_file,record_count,distinct_product_ids
products_20261007.json,25000,25000
products_20261008.json,25000,25000


Records by ingestion run:


_run_id,count
eebf67b1-6ce3-4250-bc99-e6fca60f341d,50000


In [0]:

# ============================================================
# CELL 11 — DYNAMICALLY INSPECT CLAIMED FILES IN MANIFEST
# ============================================================

from pyspark.sql import functions as F

manifest_df = spark.read.format("delta").load(MANIFEST_PATH)

# Get file names from the current run's claimed files
claimed_file_names_df = (
    claimed_files_df
    .select("file_name")
    .distinct()
)

# Inspect only the manifest records for files claimed by this run
products_manifest_df = (
    manifest_df.alias("m")
    .join(
        claimed_file_names_df.alias("c"),
        F.col("m.file_name") == F.col("c.file_name"),
        "inner"
    )
    .filter(
        (F.col("m.dataset") == DATASET_NAME)
        & (F.col("m.run_id") == MANIFEST_RUN_ID)
    )
    .select(
        F.col("m.dataset"),
        F.col("m.file_name"),
        F.col("m.status"),
        F.col("m.run_id"),
        F.col("m.source_record_count"),
        F.col("m.written_record_count"),
        F.col("m.target_path"),
        F.col("m.target_delta_version"),
        F.col("m.processed_at"),
        F.col("m.error_message")
    )
    .orderBy("file_name")
)

print("Manifest records for the current Products run:")
display(products_manifest_df)


Manifest records for the current Products run:


dataset,file_name,status,run_id,source_record_count,written_record_count,target_path,target_delta_version,processed_at,error_message
products,products_20261007.json,PROCESSING,eebf67b1-6ce3-4250-bc99-e6fca60f341d,null,null,null,null,null,null
products,products_20261008.json,PROCESSING,eebf67b1-6ce3-4250-bc99-e6fca60f341d,null,null,null,null,null,null


In [0]:

# ============================================================
# CELL 12 — UPDATE MANIFEST AFTER BRONZE VERIFICATION
# ============================================================

from delta.tables import DeltaTable
from pyspark.sql import functions as F

# Capture the current Bronze Delta version
history_df = DeltaTable.forPath(spark, BRONZE_PATH).history(1)
target_delta_version = history_df.select("version").first()["version"]

# Count records actually written for each claimed source file
written_counts_df = (
    spark.read.format("delta").load(BRONZE_PATH)
    .filter(F.col("_run_id") == MANIFEST_RUN_ID)
    .groupBy("_source_file")
    .agg(F.count("*").alias("written_record_count"))
)

# Count records read from each source file
source_counts_df = (
    bronze_df
    .groupBy("_source_file")
    .agg(F.count("*").alias("source_record_count"))
)

# Build dynamic per-file update values
updates_df = (
    claimed_files_df
    .select("dataset", "file_name", "run_id")
    .distinct()
    .join(
        source_counts_df,
        F.col("file_name") == F.col("_source_file"),
        "left"
    )
    .drop("_source_file")
    .join(
        written_counts_df,
        F.col("file_name") == F.col("_source_file"),
        "left"
    )
    .drop("_source_file")
    .withColumn("target_path", F.lit(BRONZE_PATH))
    .withColumn("target_delta_version", F.lit(target_delta_version))
    .withColumn("processed_at", F.current_timestamp())
)

display(updates_df)

# Validate counts before changing the manifest
invalid_updates = updates_df.filter(
    F.col("source_record_count").isNull()
    | F.col("written_record_count").isNull()
    | (F.col("source_record_count") != F.col("written_record_count"))
    | (F.col("written_record_count") <= 0)
)

if invalid_updates.limit(1).count() > 0:
    raise RuntimeError(
        "Manifest update cancelled: source and Bronze counts do not match."
    )

# Update only records still owned by this run and in PROCESSING state
manifest_delta = DeltaTable.forPath(spark, MANIFEST_PATH)

(
    manifest_delta.alias("m")
    .merge(
        updates_df.alias("u"),
        """
        m.dataset = u.dataset
        AND m.file_name = u.file_name
        AND m.run_id = u.run_id
        AND m.status = 'PROCESSING'
        """
    )
    .whenMatchedUpdate(set={
        "status": F.lit("SUCCESS"),
        "source_record_count": F.col("u.source_record_count"),
        "written_record_count": F.col("u.written_record_count"),
        "target_path": F.col("u.target_path"),
        "target_delta_version": F.col("u.target_delta_version"),
        "processed_at": F.col("u.processed_at"),
        "status_updated_at": F.current_timestamp(),
        "error_message": F.lit(None).cast("string")
    })
    .execute()
)

print("Manifest update command completed.")
print("Bronze Delta version recorded:", target_delta_version)


dataset,file_name,run_id,source_record_count,written_record_count,target_path,target_delta_version,processed_at
products,products_20261007.json,eebf67b1-6ce3-4250-bc99-e6fca60f341d,25000,25000,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/products/,0,2026-10-11T12:11:09.578Z
products,products_20261008.json,eebf67b1-6ce3-4250-bc99-e6fca60f341d,25000,25000,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/products/,0,2026-10-11T12:11:09.578Z


Manifest update command completed.
Bronze Delta version recorded: 0


In [0]:

# ============================================================
# CELL 13 — DIAGNOSE MANIFEST STATUS FOR CURRENT RUN
# ============================================================

from pyspark.sql import functions as F

manifest_df = spark.read.format("delta").load(MANIFEST_PATH)

print("Current run ID:", MANIFEST_RUN_ID)
print("Dataset:", DATASET_NAME)

# Inspect all Products records belonging to this run
run_manifest_df = (
    manifest_df
    .filter(
        (F.col("dataset") == DATASET_NAME)
        & (F.col("run_id") == MANIFEST_RUN_ID)
    )
    .select(
        "dataset",
        "file_name",
        "status",
        "run_id",
        "source_record_count",
        "written_record_count",
        "target_path",
        "target_delta_version",
        "processed_at",
        "error_message"
    )
    .orderBy("file_name")
)

display(run_manifest_df)

print("Manifest rows for this run:", run_manifest_df.count())

print("Status breakdown:")
display(
    run_manifest_df
    .groupBy("status")
    .count()
    .orderBy("status")
)


Current run ID: eebf67b1-6ce3-4250-bc99-e6fca60f341d
Dataset: products


dataset,file_name,status,run_id,source_record_count,written_record_count,target_path,target_delta_version,processed_at,error_message
products,products_20261007.json,SUCCESS,eebf67b1-6ce3-4250-bc99-e6fca60f341d,25000,25000,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/products/,0,2026-10-11T12:11:11.095Z,null
products,products_20261008.json,SUCCESS,eebf67b1-6ce3-4250-bc99-e6fca60f341d,25000,25000,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/products/,0,2026-10-11T12:11:11.095Z,null


Manifest rows for this run: 2
Status breakdown:


status,count
SUCCESS,2


In [0]:

# ============================================================
# CELL 14 — FINAL BRONZE / MANIFEST RECONCILIATION
# ============================================================

from pyspark.sql import functions as F

# Manifest records successfully processed in this run
successful_manifest_df = (
    spark.read.format("delta").load(MANIFEST_PATH)
    .filter(
        (F.col("dataset") == DATASET_NAME)
        & (F.col("run_id") == MANIFEST_RUN_ID)
        & (F.col("status") == "SUCCESS")
    )
    .select(
        "file_name",
        "source_record_count",
        "written_record_count",
        "target_path",
        "target_delta_version"
    )
)

# Actual Bronze counts for this run, grouped by source file
actual_bronze_counts_df = (
    spark.read.format("delta").load(BRONZE_PATH)
    .filter(F.col("_run_id") == MANIFEST_RUN_ID)
    .groupBy(F.col("_source_file").alias("file_name"))
    .agg(F.count("*").alias("actual_bronze_count"))
)

# Compare manifest and Bronze counts
reconciliation_df = (
    successful_manifest_df
    .join(actual_bronze_counts_df, on="file_name", how="left")
    .withColumn(
        "reconciliation_status",
        F.when(
            F.col("actual_bronze_count").isNull(),
            F.lit("MISSING_IN_BRONZE")
        ).when(
            F.col("written_record_count") != F.col("actual_bronze_count"),
            F.lit("COUNT_MISMATCH")
        ).otherwise(F.lit("MATCH"))
    )
    .orderBy("file_name")
)

display(reconciliation_df)

mismatch_count = reconciliation_df.filter(
    F.col("reconciliation_status") != "MATCH"
).count()

if mismatch_count > 0:
    raise RuntimeError("Bronze/manifest reconciliation failed.")

print("FINAL BRONZE RECONCILIATION PASSED")


file_name,source_record_count,written_record_count,target_path,target_delta_version,actual_bronze_count,reconciliation_status
products_20261007.json,25000,25000,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/products/,0,25000,MATCH
products_20261008.json,25000,25000,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/products/,0,25000,MATCH


FINAL BRONZE RECONCILIATION PASSED


In [0]:

# ============================================================
# CELL 15 — COMPARE CLAIMED FILES WITH BRONZE FILES
# Serverless-compatible; read-only
# ============================================================

from pyspark.sql import functions as F

# Files claimed by the current run
claimed_files_check_df = (
    spark.read.format("delta").load(MANIFEST_PATH)
    .filter(
        (F.col("dataset") == DATASET_NAME)
        & (F.col("run_id") == MANIFEST_RUN_ID)
        & (F.col("status") == "SUCCESS")
    )
    .select("file_name")
    .distinct()
)

# Files represented in Bronze
bronze_files_check_df = (
    spark.read.format("delta").load(BRONZE_PATH)
    .select(F.col("_source_file").alias("file_name"))
    .distinct()
)

comparison_df = (
    claimed_files_check_df
    .join(
        bronze_files_check_df.withColumn(
            "already_in_bronze", F.lit(True)
        ),
        on="file_name",
        how="left"
    )
    .withColumn(
        "ingestion_state",
        F.when(
            F.col("already_in_bronze").isNull(),
            F.lit("NOT_IN_BRONZE")
        ).otherwise(F.lit("ALREADY_IN_BRONZE"))
    )
    .drop("already_in_bronze")
)

display(comparison_df.orderBy("file_name"))

print("Claimed successful files:", claimed_files_check_df.count())
print("Distinct files in Bronze:", bronze_files_check_df.count())


file_name,ingestion_state
products_20261007.json,ALREADY_IN_BRONZE
products_20261008.json,ALREADY_IN_BRONZE


Claimed successful files: 2
Distinct files in Bronze: 2


In [0]:

# ============================================================
# CELL 16 — INSPECT CLAIMED FILE HASHES
# Read-only; no writes
# ============================================================

from pyspark.sql import functions as F

manifest_df = spark.read.format("delta").load(MANIFEST_PATH)
bronze_df_existing = spark.read.format("delta").load(BRONZE_PATH)

# Current run's successful files and their recorded source hashes
successful_files_df = (
    manifest_df
    .filter(
        (F.col("dataset") == DATASET_NAME)
        & (F.col("run_id") == MANIFEST_RUN_ID)
        & (F.col("status") == "SUCCESS")
    )
    .select(
        "file_name",
        "content_hash",
        "processed_content_hash",
        "written_record_count"
    )
)

# Distinct source files represented in Bronze
bronze_file_metadata_df = (
    bronze_df_existing
    .groupBy("_source_file")
    .agg(
        F.count("*").alias("bronze_record_count"),
        F.countDistinct("_run_id").alias("ingestion_run_count")
    )
    .withColumnRenamed("_source_file", "file_name")
)

display(
    successful_files_df
    .join(bronze_file_metadata_df, on="file_name", how="left")
    .orderBy("file_name")
)


file_name,content_hash,processed_content_hash,written_record_count,bronze_record_count,ingestion_run_count
products_20261007.json,70cef0bfef9126bf3502c9c40139e0f5158f0427ee440ac8cee0dfb43f083118,null,25000,25000,1
products_20261008.json,647d55b2a4e1b6a7c360248682256d5bca7c0ac36d5ce5be1d0cbaab028c0b4b,null,25000,25000,1


In [0]:

# ============================================================
# CELL 17 — CLASSIFY FILES FOR IDEMPOTENT INGESTION
# Read-only; Serverless-compatible
# ============================================================

from pyspark.sql import functions as F

manifest_df = spark.read.format("delta").load(MANIFEST_PATH)
bronze_df_existing = spark.read.format("delta").load(BRONZE_PATH)

# Current run's files that are still claimed for processing
current_claims_df = (
    manifest_df
    .filter(
        (F.col("dataset") == DATASET_NAME)
        & (F.col("run_id") == MANIFEST_RUN_ID)
        & (F.col("status") == "PROCESSING")
    )
    .select(
        "dataset",
        "file_name",
        "file_path",
        "content_hash",
        "run_id"
    )
)

# Successful manifest records for each file, regardless of run
successful_history_df = (
    manifest_df
    .filter(
        (F.col("dataset") == DATASET_NAME)
        & (F.col("status") == "SUCCESS")
    )
    .select(
        "file_name",
        F.col("processed_content_hash").alias("previous_processed_hash"),
        F.col("content_hash").alias("previous_content_hash")
    )
    .dropDuplicates(["file_name"])
)

# Compare current claims with previously successful processing
ingestion_decisions_df = (
    current_claims_df.alias("c")
    .join(
        successful_history_df.alias("h"),
        on="file_name",
        how="left"
    )
    .withColumn(
        "previous_hash",
        F.coalesce(
            F.col("previous_processed_hash"),
            F.col("previous_content_hash")
        )
    )
    .withColumn(
        "ingestion_decision",
        F.when(
            F.col("previous_hash").isNull(),
            F.lit("NEW_FILE")
        ).when(
            F.col("content_hash") == F.col("previous_hash"),
            F.lit("ALREADY_PROCESSED")
        ).otherwise(
            F.lit("CONTENT_CHANGED")
        )
    )
    .select(
        "dataset",
        "file_name",
        "content_hash",
        "previous_hash",
        "ingestion_decision",
        "run_id"
    )
)

display(ingestion_decisions_df.orderBy("file_name"))

print("Current PROCESSING claims:", current_claims_df.count())


dataset,file_name,content_hash,previous_hash,ingestion_decision,run_id


Current PROCESSING claims: 0


In [0]:

# ============================================================
# CELL 18 — DISCOVER PENDING PRODUCTS FILES
# Serverless-compatible; read-only
# ============================================================

from pyspark.sql import functions as F

manifest_df = spark.read.format("delta").load(MANIFEST_PATH)

pending_products_df = (
    manifest_df
    .filter(
        (F.col("dataset") == DATASET_NAME)
        & (F.col("status").isin("NEW", "FAILED"))
    )
    .select(
        "dataset",
        "file_name",
        "file_path",
        "status",
        "content_hash",
        "data_date",
        "retry_count"
    )
    .orderBy("data_date", "file_name")
)

print("Pending or retryable Products files:")
display(pending_products_df)

print("Pending/retryable file count:", pending_products_df.count())


Pending or retryable Products files:


dataset,file_name,file_path,status,content_hash,data_date,retry_count


Pending/retryable file count: 0


In [0]:

# ============================================================
# CELL 19 — MANIFEST HEALTH CHECK
# Read-only; Serverless-compatible
# ============================================================

from pyspark.sql import functions as F

manifest_df = spark.read.format("delta").load(MANIFEST_PATH)

print("========== MANIFEST STATUS SUMMARY ==========")

display(
    manifest_df
    .groupBy("dataset", "status")
    .count()
    .orderBy("dataset", "status")
)

print("========== FILES STILL PROCESSING ==========")

processing_df = (
    manifest_df
    .filter(F.col("status") == "PROCESSING")
    .select(
        "dataset",
        "file_name",
        "run_id",
        "claimed_at",
        "lease_expires_at",
        "last_attempt_at"
    )
    .orderBy("dataset", "file_name")
)

display(processing_df)

print("Files still PROCESSING:", processing_df.count())


========== MANIFEST STATUS SUMMARY ==========


dataset,status,count
customers,SUCCESS,3
orders,PROCESSING,1
orders,SUCCESS,1
products,SUCCESS,2


========== FILES STILL PROCESSING ==========


dataset,file_name,run_id,claimed_at,lease_expires_at,last_attempt_at
orders,orders_20261008.json,eebf67b1-6ce3-4250-bc99-e6fca60f341d,2026-10-11T11:37:52.030Z,2026-10-11T12:37:52.030Z,2026-10-11T11:37:52.030Z


Files still PROCESSING: 1


In [0]:

# ============================================================
# CELL 20 — IDEMPOTENT INCREMENTAL BRONZE WRITE
# ============================================================

from delta.tables import DeltaTable
from pyspark.sql import functions as F

expected_count = bronze_df.count()

if expected_count == 0:
    raise ValueError("Bronze DataFrame is empty. Write cancelled.")

# Ensure one source row per file + product ID
duplicate_keys_df = (
    bronze_df
    .groupBy("_source_file", "product_id")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_keys_df.limit(1).count() > 0:
    raise ValueError(
        "Duplicate source-file/product-ID keys found. "
        "Resolve duplicates before writing."
    )

if not DeltaTable.isDeltaTable(spark, BRONZE_PATH):
    print("Bronze Delta table does not exist. Creating it.")

    (
        bronze_df.write
        .format("delta")
        .mode("append")
        .save(BRONZE_PATH)
    )

else:
    print("Bronze Delta table exists. Running incremental MERGE.")

    target = DeltaTable.forPath(spark, BRONZE_PATH)

    (
        target.alias("t")
        .merge(
            bronze_df.alias("s"),
            """
            t._source_file = s._source_file
            AND t.product_id = s.product_id
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

print("Incremental Bronze MERGE completed.")

# Verify the source-file/product-ID keys are unique in the target
written_df = spark.read.format("delta").load(BRONZE_PATH)

duplicate_target_keys_df = (
    written_df
    .groupBy("_source_file", "product_id")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_target_keys_df.limit(1).count() > 0:
    raise RuntimeError(
        "Duplicate keys found in Bronze after MERGE."
    )

print("Target record count:", written_df.count())
print("Records by source file:")

display(
    written_df
    .groupBy("_source_file")
    .count()
    .orderBy("_source_file")
)


Bronze Delta table exists. Running incremental MERGE.
Incremental Bronze MERGE completed.
Target record count: 50000
Records by source file:


_source_file,count
products_20261007.json,25000
products_20261008.json,25000


In [0]:

# ============================================================
# CELL 21 — RECORD PROCESSED CONTENT HASH
# ============================================================

from delta.tables import DeltaTable
from pyspark.sql import functions as F

manifest_df = spark.read.format("delta").load(MANIFEST_PATH)
bronze_df_existing = spark.read.format("delta").load(BRONZE_PATH)

# Successful Products files for the current run
successful_files_df = (
    manifest_df
    .filter(
        (F.col("dataset") == DATASET_NAME)
        & (F.col("run_id") == MANIFEST_RUN_ID)
        & (F.col("status") == "SUCCESS")
    )
    .select(
        "dataset",
        "file_name",
        "run_id",
        "content_hash",
        "written_record_count"
    )
)

# Independently count the persisted Bronze records
actual_counts_df = (
    bronze_df_existing
    .filter(F.col("_run_id") == MANIFEST_RUN_ID)
    .groupBy(F.col("_source_file").alias("file_name"))
    .agg(F.count("*").alias("actual_bronze_count"))
)

hash_updates_df = (
    successful_files_df
    .join(actual_counts_df, on="file_name", how="inner")
    .filter(
        F.col("content_hash").isNotNull()
        & (
            F.col("written_record_count")
            == F.col("actual_bronze_count")
        )
    )
    .select(
        "dataset",
        "file_name",
        "run_id",
        F.col("content_hash").alias("processed_content_hash")
    )
)

display(hash_updates_df)

# Update only matching successful records for this run
manifest_delta = DeltaTable.forPath(spark, MANIFEST_PATH)

(
    manifest_delta.alias("m")
    .merge(
        hash_updates_df.alias("u"),
        """
        m.dataset = u.dataset
        AND m.file_name = u.file_name
        AND m.run_id = u.run_id
        AND m.status = 'SUCCESS'
        """
    )
    .whenMatchedUpdate(set={
        "processed_content_hash": F.col("u.processed_content_hash")
    })
    .execute()
)

print("Processed content hash update completed.")



dataset,file_name,run_id,processed_content_hash
products,products_20261007.json,eebf67b1-6ce3-4250-bc99-e6fca60f341d,70cef0bfef9126bf3502c9c40139e0f5158f0427ee440ac8cee0dfb43f083118
products,products_20261008.json,eebf67b1-6ce3-4250-bc99-e6fca60f341d,647d55b2a4e1b6a7c360248682256d5bca7c0ac36d5ce5be1d0cbaab028c0b4b


Processed content hash update completed.


In [0]:

# ============================================================
# CELL 22 — FINAL PRODUCTS PIPELINE VERIFICATION
# Serverless-compatible; read-only
# ============================================================

from pyspark.sql import functions as F

manifest_df = spark.read.format("delta").load(MANIFEST_PATH)
bronze_df_existing = spark.read.format("delta").load(BRONZE_PATH)

# Manifest records for this Products run
final_manifest_df = (
    manifest_df
    .filter(
        (F.col("dataset") == DATASET_NAME)
        & (F.col("run_id") == MANIFEST_RUN_ID)
    )
    .select(
        "file_name",
        "status",
        "source_record_count",
        "written_record_count",
        "content_hash",
        "processed_content_hash",
        "target_delta_version"
    )
)

# Actual Bronze counts for this run
actual_counts_df = (
    bronze_df_existing
    .filter(F.col("_run_id") == MANIFEST_RUN_ID)
    .groupBy(F.col("_source_file").alias("file_name"))
    .agg(F.count("*").alias("actual_bronze_count"))
)

# Reconcile manifest and Bronze
final_check_df = (
    final_manifest_df
    .join(actual_counts_df, on="file_name", how="left")
    .withColumn(
        "validation_status",
        F.when(
            F.col("status") != "SUCCESS",
            F.lit("STATUS_MISMATCH")
        ).when(
            F.col("content_hash").isNull()
            | F.col("processed_content_hash").isNull()
            | (
                F.col("content_hash")
                != F.col("processed_content_hash")
            ),
            F.lit("HASH_MISMATCH")
        ).when(
            F.col("source_record_count")
            != F.col("written_record_count"),
            F.lit("MANIFEST_COUNT_MISMATCH")
        ).when(
            F.col("written_record_count")
            != F.col("actual_bronze_count"),
            F.lit("BRONZE_COUNT_MISMATCH")
        ).otherwise(F.lit("PASS"))
    )
    .orderBy("file_name")
)

display(final_check_df)

total_files = final_check_df.count()

failed_files = final_check_df.filter(
    F.col("validation_status") != "PASS"
).count()

print("Files checked:", total_files)
print("Files failing validation:", failed_files)
print(
    "Total Bronze records for current run:",
    bronze_df_existing
    .filter(F.col("_run_id") == MANIFEST_RUN_ID)
    .count()
)

if total_files == 0 or failed_files > 0:
    raise RuntimeError("Final Products pipeline verification failed.")

print("PRODUCTS PIPELINE FINAL VERIFICATION PASSED!")


file_name,status,source_record_count,written_record_count,content_hash,processed_content_hash,target_delta_version,actual_bronze_count,validation_status
products_20261007.json,SUCCESS,25000,25000,70cef0bfef9126bf3502c9c40139e0f5158f0427ee440ac8cee0dfb43f083118,70cef0bfef9126bf3502c9c40139e0f5158f0427ee440ac8cee0dfb43f083118,0,25000,PASS
products_20261008.json,SUCCESS,25000,25000,647d55b2a4e1b6a7c360248682256d5bca7c0ac36d5ce5be1d0cbaab028c0b4b,647d55b2a4e1b6a7c360248682256d5bca7c0ac36d5ce5be1d0cbaab028c0b4b,0,25000,PASS


Files checked: 2
Files failing validation: 0
Total Bronze records for current run: 50000
PRODUCTS PIPELINE FINAL VERIFICATION PASSED!
